# Gen Z Survival Dashboard 2026
## Steps 1–2: Data Collection & SQL Setup

### Real Data Sources
| Source | File | Granularity |
|--------|------|-------------|
| Numbeo Cost of Living + QoL | `numbeo_merged.csv` | City-level, 30 cities |
| World Bank API | automated | Country-level, latest available year |
| Stack Overflow Survey 2024 | `survey_results_public.csv` | 65k responses, country-level |
| LinkedIn Job Postings (Kaggle) | `postings.csv` | ~124k postings, mostly US |
| Google Trends (pytrends) | automated | Region-level, 2023–2024 avg |

### Documented Limitations
- LinkedIn dataset is US-centric. Non-US cities (< 20 postings) get `data_reliable=False` and are filled via regional median of reliable cities.
- World Bank and SO Survey are country-level; cities in the same country share those values.
- `tech_adoption.csv` from the original pipeline was **synthetic** and has been dropped entirely — replaced by SO Survey.

In [5]:
# !pip install pandas numpy requests openpyxl sqlalchemy matplotlib seaborn scikit-learn scipy pytrends tqdm
import pandas as pd
import numpy as np
import requests
import sqlite3
import os
import time
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')
for folder in ['data/raw', 'data/processed', 'sql', 'visualizations', 'documentation']:
    os.makedirs(folder, exist_ok=True)

# ── Set this to wherever you saved the downloaded files ──────────────────────
DATA_DIR = 'raw_uploads/'
os.makedirs(DATA_DIR, exist_ok=True)
print('Setup complete.')

Setup complete.


## 1.1 City Master List

In [6]:
# li_pattern: substring matched against LinkedIn 'location' column
# so_country: exact country name in Stack Overflow survey
# wb_code:    ISO 3166-1 alpha-2 for World Bank API

cities_meta = [
    ('New York',      'USA',         'North America',  8800000,  'US', 'United States of America',                            'New York, NY'),
    ('San Francisco', 'USA',         'North America',  873965,   'US', 'United States of America',                            'San Francisco, CA'),
    ('Austin',        'USA',         'North America',  978908,   'US', 'United States of America',                            'Austin, TX'),
    ('Seattle',       'USA',         'North America',  753675,   'US', 'United States of America',                            'Seattle, WA'),
    ('Boston',        'USA',         'North America',  675647,   'US', 'United States of America',                            'Boston, MA'),
    ('London',        'UK',          'Europe',         9002488,  'GB', 'United Kingdom of Great Britain and Northern Ireland', 'London'),
    ('Berlin',        'Germany',     'Europe',         3644826,  'DE', 'Germany',                                             'Berlin'),
    ('Amsterdam',     'Netherlands', 'Europe',         872680,   'NL', 'Netherlands',                                         'Amsterdam'),
    ('Stockholm',     'Sweden',      'Europe',         975551,   'SE', 'Sweden',                                              'Stockholm'),
    ('Copenhagen',    'Denmark',     'Europe',         623404,   'DK', 'Denmark',                                             'Copenhagen'),
    ('Singapore',     'Singapore',   'Asia',           5453600,  'SG', 'Singapore',                                           'Singapore'),
    ('Tokyo',         'Japan',       'Asia',           13960000, 'JP', 'Japan',                                               'Tokyo'),
    ('Seoul',         'South Korea', 'Asia',           9776000,  'KR', 'Republic of Korea',                                   'Seoul'),
    ('Sydney',        'Australia',   'Oceania',        5312163,  'AU', 'Australia',                                           'Sydney'),
    ('Melbourne',     'Australia',   'Oceania',        5078193,  'AU', 'Australia',                                           'Melbourne'),
    ('Toronto',       'Canada',      'North America',  2930000,  'CA', 'Canada',                                              'Toronto'),
    ('Vancouver',     'Canada',      'North America',  675218,   'CA', 'Canada',                                              'Vancouver'),
    ('Montreal',      'Canada',      'North America',  1762949,  'CA', 'Canada',                                              'Montreal'),
    ('Barcelona',     'Spain',       'Europe',         1636762,  'ES', 'Spain',                                               'Barcelona'),
    ('Lisbon',        'Portugal',    'Europe',         504718,   'PT', 'Portugal',                                            'Lisbon'),
    ('Dublin',        'Ireland',     'Europe',         1173179,  'IE', 'Ireland',                                             'Dublin'),
    ('Edinburgh',     'UK',          'Europe',         524930,   'GB', 'United Kingdom of Great Britain and Northern Ireland', 'Edinburgh'),
    ('Paris',         'France',      'Europe',         2161000,  'FR', 'France',                                              'Paris'),
    ('Munich',        'Germany',     'Europe',         1471508,  'DE', 'Germany',                                             'Munich'),
    ('Zurich',        'Switzerland', 'Europe',         402762,   'CH', 'Switzerland',                                         'Zurich'),
    ('Tel Aviv',      'Israel',      'Middle East',    460613,   'IL', 'Israel',                                              'Tel Aviv'),
    ('Dubai',         'UAE',         'Middle East',    3411200,  'AE', 'United Arab Emirates',                                'Dubai'),
    ('Bangalore',     'India',       'Asia',           12765000, 'IN', 'India',                                               'Bangalore'),
    ('Mumbai',        'India',       'Asia',           12442373, 'IN', 'India',                                               'Mumbai'),
    ('Mexico City',   'Mexico',      'Latin America',  9209944,  'MX', 'Mexico',                                              'Mexico City'),
]

df_cities = pd.DataFrame(cities_meta, columns=[
    'city','country','region','population','wb_code','so_country','li_pattern'
])
df_cities['city_id'] = range(1, len(df_cities) + 1)
print(f'{len(df_cities)} cities loaded.')
print(df_cities[['city_id','city','country','region']].to_string(index=False))

30 cities loaded.
 city_id          city     country        region
       1      New York         USA North America
       2 San Francisco         USA North America
       3        Austin         USA North America
       4       Seattle         USA North America
       5        Boston         USA North America
       6        London          UK        Europe
       7        Berlin     Germany        Europe
       8     Amsterdam Netherlands        Europe
       9     Stockholm      Sweden        Europe
      10    Copenhagen     Denmark        Europe
      11     Singapore   Singapore          Asia
      12         Tokyo       Japan          Asia
      13         Seoul South Korea          Asia
      14        Sydney   Australia       Oceania
      15     Melbourne   Australia       Oceania
      16       Toronto      Canada North America
      17     Vancouver      Canada North America
      18      Montreal      Canada North America
      19     Barcelona       Spain        Europe
  

## 1.2 Numbeo — Cost of Living & Quality of Life (pre-merged)

In [7]:
df_numbeo = pd.read_csv(os.path.join(DATA_DIR, 'numbeo_merged.csv'))

missing = set(df_cities['city']) - set(df_numbeo['city'])
print(f'Missing cities: {missing if missing else "None — all 30 present"}')
print(f'Shape: {df_numbeo.shape}')
print(df_numbeo.to_string(index=False))

Missing cities: None — all 30 present
Shape: (30, 8)
 city_id          city  cost_of_living_index  rent_index  purchasing_power_index  quality_of_life_index  healthcare_index  pollution_index
       1      New York                 100.0       100.0                   100.0                  136.5              62.3             57.6
       2 San Francisco                  99.6        92.4                   104.0                  140.7              63.9             50.4
       3        Austin                  73.2        66.4                   128.3                  177.9              67.6             40.4
       4       Seattle                  92.9        68.0                   126.2                  176.9              72.2             30.9
       5        Boston                  84.6        75.7                   111.1                  168.8              74.5             30.9
       6        London                  74.2        63.8                    88.1                  131.5          

## 1.3 World Bank API — Youth Unemployment, Inflation, GDP

In [8]:
WB_INDICATORS = {
    'SL.UEM.1524.ZS': 'youth_unemployment_rate',
    'FP.CPI.TOTL.ZG': 'inflation_rate',
    'NY.GDP.PCAP.CD':  'gdp_per_capita_usd',
}
WB_CACHE = 'data/raw/world_bank_indicators.csv'

def fetch_wb(indicator, wb_code, retries=3):
    url = (f'https://api.worldbank.org/v2/country/{wb_code}'
           f'/indicator/{indicator}?format=json&mrv=3')
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=10)
            records = r.json()[1]
            if records:
                for rec in records:
                    if rec['value'] is not None:
                        return round(rec['value'], 3), rec['date']
        except Exception as e:
            if attempt == retries - 1:
                print(f'  Failed {wb_code}/{indicator}: {e}')
        time.sleep(0.4)
    return None, None

if os.path.exists(WB_CACHE):
    print(f'Loading cached World Bank data...')
    df_wb = pd.read_csv(WB_CACHE)
else:
    unique_countries = df_cities[['country','wb_code']].drop_duplicates()
    rows = []
    print('Fetching World Bank API...')
    for _, row in tqdm(unique_countries.iterrows(), total=len(unique_countries)):
        rec = {'country': row['country'], 'wb_code': row['wb_code']}
        for ind, col in WB_INDICATORS.items():
            val, yr = fetch_wb(ind, row['wb_code'])
            rec[col] = val
            rec[f'{col}_year'] = yr
        rows.append(rec)
    df_wb = pd.DataFrame(rows)
    df_wb.to_csv(WB_CACHE, index=False)
    print('✅ World Bank data saved.')

print(df_wb[['country','youth_unemployment_rate','inflation_rate','gdp_per_capita_usd']].to_string(index=False))

Loading cached World Bank data...
    country  youth_unemployment_rate  inflation_rate  gdp_per_capita_usd
        USA                    9.340           2.950           84534.041
         UK                   14.647           3.272           53246.368
    Germany                    6.862           2.256           56103.732
Netherlands                    8.833           3.348           67520.422
     Sweden                   24.312           2.836           57117.488
    Denmark                   11.737           1.372           71026.483
  Singapore                    6.781           2.390           90674.067
      Japan                    3.898           2.739           32487.078
South Korea                    6.695           2.322           36238.640
  Australia                    9.610           3.167           64603.986
     Canada                   13.800           2.382           54340.348
      Spain                   24.745           2.774           35326.768
   Portugal      

## 1.4 Stack Overflow Developer Survey 2024

Actual column names in `survey_results_public.csv`:
- `AISelect` — `'Yes'` / `'No, and I don't plan to'` / `'No, but I plan to soon'`
- `AIToolCurrently Using` — semicolon-separated tool names (e.g. `'ChatGPT;GitHub Copilot'`)
- `RemoteWork` — `'Remote'` / `'Hybrid (some remote, some in-person)'` / `'In-person'`
- `Country` — full country name matching `so_country` in city master

In [9]:
SO_FILE  = os.path.join(DATA_DIR, 'survey_results_public.csv')
SO_CACHE = 'data/raw/so_survey_by_city.csv'

if os.path.exists(SO_CACHE):
    print('Loading cached SO data...')
    df_so_city = pd.read_csv(SO_CACHE)
else:
    SO_COLS = ['ResponseId', 'Country', 'AISelect',
               'AIToolCurrently Using', 'RemoteWork']
    df_so = pd.read_csv(SO_FILE, usecols=SO_COLS, low_memory=False)
    print(f'SO Survey: {len(df_so):,} responses, {df_so["Country"].nunique()} countries')

    df_so['uses_ai']            = (df_so['AISelect'] == 'Yes').astype(int)
    df_so['uses_chatgpt']       = df_so['AIToolCurrently Using'].str.contains('ChatGPT', na=False).astype(int)
    df_so['uses_copilot']       = df_so['AIToolCurrently Using'].str.contains('GitHub Copilot', na=False).astype(int)
    df_so['is_remote_hybrid']   = df_so['RemoteWork'].isin(
        ['Remote', 'Hybrid (some remote, some in-person)']).astype(int)
    df_so['is_fully_remote']    = (df_so['RemoteWork'] == 'Remote').astype(int)

    df_so_country = (
        df_so.groupby('Country')
        .agg(
            so_respondents      = ('ResponseId',       'count'),
            ai_tool_usage_rate  = ('uses_ai',          lambda x: round(x.mean()*100, 2)),
            chatgpt_usage_pct   = ('uses_chatgpt',     lambda x: round(x.mean()*100, 2)),
            copilot_usage_pct   = ('uses_copilot',     lambda x: round(x.mean()*100, 2)),
            remote_hybrid_pct   = ('is_remote_hybrid', lambda x: round(x.mean()*100, 2)),
            fully_remote_pct    = ('is_fully_remote',  lambda x: round(x.mean()*100, 2)),
        )
        .reset_index()
        .query('so_respondents >= 50')
        .rename(columns={'Country': 'so_country'})
    )

    df_so_city = df_cities[['city_id','city','so_country']].merge(
        df_so_country, on='so_country', how='left')

    missing_so = df_so_city[df_so_city['ai_tool_usage_rate'].isna()]['city'].tolist()
    if missing_so:
        print(f'⚠️  No SO data for: {missing_so}')

    df_so_city.to_csv(SO_CACHE, index=False)
    print('✅ SO data saved.')

print(df_so_city[['city','so_respondents','ai_tool_usage_rate',
                   'chatgpt_usage_pct','remote_hybrid_pct']]
      .sort_values('ai_tool_usage_rate', ascending=False).to_string(index=False))

Loading cached SO data...
         city  so_respondents  ai_tool_usage_rate  chatgpt_usage_pct  remote_hybrid_pct
     Tel Aviv             604               72.02                0.0              66.72
        Dubai             130               69.23                0.0              45.38
    Bangalore            4231               66.82                0.0              48.43
       Mumbai            4231               66.82                0.0              48.43
    Singapore             177               66.67                0.0              58.19
       Lisbon             470               65.32                0.0              76.17
        Seoul              57               64.91                0.0              28.07
    Barcelona            1123               64.83                0.0              76.31
  Mexico City             402               63.93                0.0              68.16
        Tokyo             288               62.50                0.0              71.88
      

## 1.5 LinkedIn Job Postings (Kaggle — `postings.csv`)

Actual column names:
- `title` — job title string
- `location` — `'City, ST'` format (US) or city name (international)
- `remote_allowed` — `1.0` = remote allowed, `NaN` = not remote
- `formatted_work_type` — `'Full-time'`, `'Part-time'`, `'Contract'`, etc.
- `normalized_salary` — annualised USD salary

**Coverage note:** Dataset is 123,849 postings, ~95% US locations.
International cities have very low counts and are flagged `data_reliable=False`.

In [10]:
LI_FILE  = os.path.join(DATA_DIR, 'postings.csv')
LI_CACHE = 'data/raw/linkedin_job_market.csv'

AI_KEYWORDS = [
    'machine learning', 'artificial intelligence', r'\bai\b',
    'data scientist', 'nlp', 'deep learning', r'\bllm\b',
    'computer vision', 'mlops', 'ml engineer', 'research scientist',
    'data science', 'generative ai'
]
MIN_RELIABLE = 20

if os.path.exists(LI_CACHE):
    print('Loading cached LinkedIn data...')
    df_jobs = pd.read_csv(LI_CACHE)
else:
    print('Loading postings.csv (~30s)...')
    df_li = pd.read_csv(
        LI_FILE,
        usecols=['title','location','remote_allowed',
                 'formatted_work_type','normalized_salary'],
        low_memory=False
    )
    print(f'Loaded {len(df_li):,} postings')

    title_lower = df_li['title'].str.lower().fillna('')
    df_li['is_ai']    = title_lower.str.contains('|'.join(AI_KEYWORDS), regex=True).astype(int)
    df_li['is_remote'] = (df_li['remote_allowed'] == 1.0).astype(int)
    loc_lower = df_li['location'].str.lower().fillna('')

    rows = []
    for _, city_row in df_cities.iterrows():
        pattern = city_row['li_pattern'].lower()
        mask    = loc_lower.str.contains(pattern, regex=False)
        sub     = df_li[mask]
        n       = len(sub)
        rows.append({
            'city_id':            city_row['city_id'],
            'city':               city_row['city'],
            'total_job_postings': n,
            'ai_job_postings':    int(sub['is_ai'].sum()),
            'remote_job_ratio':   round(sub['is_remote'].mean()*100, 2) if n > 0 else np.nan,
            'ai_pct_of_total':    round(sub['is_ai'].mean()*100, 2)    if n > 0 else np.nan,
            'median_salary_usd':  round(sub['normalized_salary'].median(), 0) if n > 0 else np.nan,
            'data_reliable':      n >= MIN_RELIABLE,
        })

    df_jobs = pd.DataFrame(rows)
    df_jobs.to_csv(LI_CACHE, index=False)
    print('✅ LinkedIn data saved.')

unreliable = df_jobs[~df_jobs['data_reliable']]['city'].tolist()
print(f'⚠️  Unreliable (< {MIN_RELIABLE} postings): {unreliable}')
print(f'   These will be filled via regional median in Step 1.7.')
print()
print(df_jobs.sort_values('ai_job_postings', ascending=False).to_string(index=False))

Loading cached LinkedIn data...
⚠️  Unreliable (< 20 postings): ['Amsterdam', 'Stockholm', 'Copenhagen', 'Singapore', 'Tokyo', 'Seoul', 'Sydney', 'Toronto', 'Montreal', 'Barcelona', 'Lisbon', 'Edinburgh', 'Paris', 'Munich', 'Zurich', 'Tel Aviv', 'Dubai', 'Bangalore', 'Mumbai', 'Mexico City']
   These will be filled via regional median in Step 1.7.

 city_id          city  total_job_postings  ai_job_postings  remote_job_ratio  ai_pct_of_total  median_salary_usd  data_reliable
       1      New York                2756               46              5.62             1.67           110000.0           True
       2 San Francisco                 967               31              8.89             3.21           130000.0           True
       4       Seattle                 819               19              7.57             2.32           125000.0           True
       5        Boston                1201               18              9.58             1.50           104000.0           True
    

## 1.6 Google Trends — Burnout, Side Hustle, Work-Life Balance

In [11]:
import subprocess
subprocess.run(['pip', 'install', 'urllib3<2.0', '--quiet'], check=True)
print('✅ urllib3 downgraded. Now restart your kernel, then re-run from cell 1.6.')

✅ urllib3 downgraded. Now restart your kernel, then re-run from cell 1.6.


In [15]:
# ── 1.6 Lifestyle Proxies from Stack Overflow Survey 2024 ────────────────────
# pytrends has a urllib3 v2 incompatibility. Instead we derive lifestyle signals
# directly from SO Survey microdata — a more defensible source (65k respondents).
#
# Column names reflect EXACTLY what is measured, not higher-level concepts:
#   dev_frustration_pct     → % devs reporting 4+ distinct work frustrations
#                             (technical debt, slow tools, unclear requirements…)
#                             Source: Frustration column (semicolon-separated list)
#   job_dissatisfaction_pct → % devs with JobSat score 0–4 (out of 10)
#   job_satisfaction_pct    → % devs with JobSat score 7–10 (out of 10)
#   ⚠️  job_dissatisfaction_pct and job_satisfaction_pct are near-perfect inverses
#       (both from JobSat). Don't include both in the same model.
#   hobby_coding_pct        → % devs who code as a hobby outside work
#
# Dropped vs. original plan:
#   remote_work  — fully_remote_pct already collected in step 1.4 (duplicate)
#   ai_interest  — ai_tool_usage_rate already collected in step 1.4 (duplicate)

TRENDS_CACHE = 'data/raw/google_trends.csv'
SO_FILE = os.path.join(DATA_DIR, 'survey_results_public.csv')

SO_PROXY_COLS = ['ResponseId', 'Country', 'JobSat', 'Frustration', 'CodingActivities']
df_so_proxy_raw = pd.read_csv(SO_FILE, usecols=SO_PROXY_COLS, low_memory=False)
print(f'SO Survey rows: {len(df_so_proxy_raw):,}')

# dev_frustration_pct: Frustration is a SEMICOLON-SEPARATED LIST of work frustrations
# (e.g. 'Dealing with tech debt;Slow build/test cycles;Inadequate tooling').
# Count of items per respondent = proxy for overall work stress level.
# Threshold: 4+ distinct frustrations → flagged as highly frustrated.
def count_frustrations(val):
    if pd.isna(val) or val == 'None of these':
        return 0
    return len(str(val).split(';'))

df_so_proxy_raw['frustration_count'] = df_so_proxy_raw['Frustration'].apply(count_frustrations)
df_so_proxy_raw['high_frustration']  = (df_so_proxy_raw['frustration_count'] >= 4).astype(int)

# job_dissatisfaction_pct / job_satisfaction_pct from numeric JobSat (0–10)
df_so_proxy_raw['job_dissatisfied'] = (
    pd.to_numeric(df_so_proxy_raw['JobSat'], errors='coerce') <= 4
).astype(int)

df_so_proxy_raw['job_satisfied'] = (
    pd.to_numeric(df_so_proxy_raw['JobSat'], errors='coerce') >= 7
).astype(int)

# hobby_coding_pct
df_so_proxy_raw['codes_hobby'] = df_so_proxy_raw['CodingActivities'].str.contains(
    'Hobby', na=False).astype(int)

# ── Aggregate by country ──────────────────────────────────────────────────────
df_proxy_country = (
    df_so_proxy_raw
    .groupby('Country')
    .agg(
        n                       = ('ResponseId',      'count'),
        dev_frustration_pct     = ('high_frustration', lambda x: round(x.mean() * 100, 2)),
        job_dissatisfaction_pct = ('job_dissatisfied', lambda x: round(x.mean() * 100, 2)),
        job_satisfaction_pct    = ('job_satisfied',    lambda x: round(x.mean() * 100, 2)),
        hobby_coding_pct        = ('codes_hobby',      lambda x: round(x.mean() * 100, 2)),
    )
    .reset_index()
    .query('n >= 50')
    .rename(columns={'Country': 'so_country'})
)

print('Sample country-level values:')
print(df_proxy_country.head(10)[['so_country','dev_frustration_pct',
    'job_dissatisfaction_pct','job_satisfaction_pct','hobby_coding_pct']].to_string(index=False))

corr_check = df_proxy_country[['job_dissatisfaction_pct','job_satisfaction_pct']].corr()
print(f"\njob_dissatisfaction vs job_satisfaction r = {corr_check.iloc[0,1]:.3f}  "
      "(near-perfect inverse — as expected from the same source column)")

# ── Map to cities ─────────────────────────────────────────────────────────────
df_trends = df_cities[['city_id', 'city', 'so_country']].merge(
    df_proxy_country[['so_country', 'dev_frustration_pct', 'job_dissatisfaction_pct',
                       'job_satisfaction_pct', 'hobby_coding_pct']],
    on='so_country', how='left'
)
df_trends = df_trends[['city', 'dev_frustration_pct', 'job_dissatisfaction_pct',
                        'job_satisfaction_pct', 'hobby_coding_pct']]

missing = df_trends[df_trends['dev_frustration_pct'].isna()]['city'].tolist()
if missing:
    print(f'\n⚠️  No proxy data for: {missing}')

print('\nFinal proxy data:')
print(df_trends.to_string(index=False))
df_trends.to_csv(TRENDS_CACHE, index=False)
print('\n✅ Saved')


SO Survey rows: 65,437
Frustration values: {'None of these': 2364, 'Amount of technical debt': 2067, 'Amount of technical debt;Reliability of tools/systems used in work': 831, 'Amount of technical debt;Complexity of tech stack for deployment;Complexity of tech stack for build': 666, 'Amount of technical debt;Tracking my work': 556, 'Amount of technical debt;Patching/updating core components': 549, 'Amount of technical debt;Complexity of tech stack for build': 495, 'Amount of technical debt;Complexity of tech stack for deployment': 454, 'Tracking my work': 436, 'Complexity of tech stack for deployment;Complexity of tech stack for build': 397, 'Tracking my work;Showing my contributions': 377, 'Amount of technical debt;Tracking my work;Showing my contributions': 346, 'Amount of technical debt;Number of software tools in use': 341, 'Amount of technical debt;Complexity of tech stack for deployment;Complexity of tech stack for build;Reliability of tools/systems used in work': 334, 'Reliabili

## 1.7 Merge All Sources → Master Dataset

In [16]:
df_master = df_cities.copy()

# Numbeo (city-level)
df_master = df_master.merge(
    df_numbeo.drop(columns=['city'], errors='ignore'), on='city_id', how='left')

# World Bank (country-level via wb_code)
df_master = df_master.merge(
    df_wb[['wb_code','youth_unemployment_rate','inflation_rate','gdp_per_capita_usd']],
    on='wb_code', how='left')

# Stack Overflow (country-level via city_id)
df_master = df_master.merge(
    df_so_city[['city_id','so_respondents','ai_tool_usage_rate','chatgpt_usage_pct',
                'copilot_usage_pct','remote_hybrid_pct','fully_remote_pct']],
    on='city_id', how='left')

# LinkedIn (city-level, includes data_reliable)
df_master = df_master.merge(
    df_jobs[['city_id','total_job_postings','ai_job_postings','remote_job_ratio',
             'ai_pct_of_total','median_salary_usd','data_reliable']],
    on='city_id', how='left')

# Google Trends (city-level)
df_master = df_master.merge(df_trends, on='city', how='left')

print('MERGE COMPLETE')
print(f'Shape: {df_master.shape}')
missing = df_master.isnull().sum()
print('\nMissing values:')
print(missing[missing > 0].to_string())

df_master.to_csv('data/raw/master_raw_merged.csv', index=False)

MERGE COMPLETE
Shape: (30, 35)

Missing values:
remote_job_ratio     13
ai_pct_of_total      13
median_salary_usd    15
remote_work          30
ai_interest          30


## 1.8 Missing Value Handling — Fully Documented

In [18]:
df_clean = df_master.copy()
imputation_log = []

def impute_regional(df, col, group_col='region'):
    n = df[col].isna().sum()
    if n == 0: return df
    df[col] = df[col].fillna(df.groupby(group_col)[col].transform('median'))
    df[col] = df[col].fillna(df[col].median())  # global fallback
    imputation_log.append({'column': col, 'n_imputed': n, 'method': f'regional median (group={group_col})'})
    return df

# Numbeo — regional median
for col in ['cost_of_living_index','rent_index','purchasing_power_index',
            'quality_of_life_index','healthcare_index','pollution_index']:
    df_clean = impute_regional(df_clean, col)

# World Bank — regional median
for col in ['youth_unemployment_rate','inflation_rate','gdp_per_capita_usd']:
    df_clean = impute_regional(df_clean, col)

# SO Survey — regional median
for col in ['ai_tool_usage_rate','chatgpt_usage_pct','copilot_usage_pct',
            'remote_hybrid_pct','fully_remote_pct']:
    df_clean = impute_regional(df_clean, col)

# LinkedIn — for unreliable cities only, fill from regional median of reliable cities
reliable_mask   = df_clean['data_reliable'].fillna(False).astype(bool)
unreliable_mask = ~reliable_mask

for col in ['total_job_postings', 'ai_job_postings', 'remote_job_ratio',
            'ai_pct_of_total', 'median_salary_usd']:
    df_clean[col] = df_clean[col].astype(float)
    n = unreliable_mask.sum()
    reg_meds = df_clean[reliable_mask].groupby('region')[col].median()
    for idx in df_clean[unreliable_mask].index:
        region = df_clean.loc[idx, 'region']
        fallback = df_clean.loc[reliable_mask, col].median()
        df_clean.loc[idx, col] = reg_meds.get(region, fallback)
    imputation_log.append({'column': col, 'n_imputed': n,
                           'method': 'regional median of reliable cities (LinkedIn intl gap)'})

# SO Survey lifestyle proxies — regional median
# job_dissatisfaction_pct and job_satisfaction_pct are near-perfect inverses;
# imputing independently is fine since they are used separately in models.
for col in ['dev_frustration_pct','job_dissatisfaction_pct',
            'job_satisfaction_pct','hobby_coding_pct']:
    df_clean = impute_regional(df_clean, col)

remaining_nulls = df_clean.select_dtypes(include='number').isnull().sum().sum()
print('IMPUTATION LOG')
print(pd.DataFrame(imputation_log).to_string(index=False))
print(f'\nRemaining numeric nulls: {remaining_nulls}')

df_clean.to_csv('data/processed/master_clean.csv', index=False)
print('\n✅ Clean master saved to data/processed/master_clean.csv')


IMPUTATION LOG
            column  n_imputed                                                 method
total_job_postings         20 regional median of reliable cities (LinkedIn intl gap)
   ai_job_postings         20 regional median of reliable cities (LinkedIn intl gap)
  remote_job_ratio         20 regional median of reliable cities (LinkedIn intl gap)
   ai_pct_of_total         20 regional median of reliable cities (LinkedIn intl gap)
 median_salary_usd         20 regional median of reliable cities (LinkedIn intl gap)
       remote_work         30                         regional median (group=region)
       ai_interest         30                         regional median (group=region)

Remaining numeric nulls: 60

✅ Clean master saved to data/processed/master_clean.csv


---
## Step 2: SQL Database

In [19]:
conn = sqlite3.connect('data/gen_z_survival.db')

TABLE_COLS = {
    'cities': ['city_id','city','country','region','population'],
    'economic_indicators': ['city_id','cost_of_living_index','rent_index',
        'purchasing_power_index','quality_of_life_index','healthcare_index',
        'pollution_index','youth_unemployment_rate','inflation_rate','gdp_per_capita_usd'],
    'job_market': ['city_id','total_job_postings','ai_job_postings',
        'remote_job_ratio','ai_pct_of_total','median_salary_usd','data_reliable'],
    'tech_adoption': ['city_id','ai_tool_usage_rate','chatgpt_usage_pct',
        'copilot_usage_pct','remote_hybrid_pct','fully_remote_pct'],
    # Column names reflect exactly what is measured (SO Survey proxies).
    # See step 1.6 for proxy definitions.
    'lifestyle_metrics': ['city_id','dev_frustration_pct','job_dissatisfaction_pct',
        'job_satisfaction_pct','hobby_coding_pct'],
}

for tbl, cols in TABLE_COLS.items():
    df_clean[cols].to_sql(tbl, conn, if_exists='replace', index=False)
    n = pd.read_sql(f'SELECT COUNT(*) AS n FROM {tbl}', conn)['n'][0]
    print(f'  {tbl}: {n} rows')

conn.close()
print('\n✅ SQLite DB ready: data/gen_z_survival.db')


  cities: 30 rows
  economic_indicators: 30 rows
  job_market: 30 rows
  tech_adoption: 30 rows
  lifestyle_metrics: 30 rows

✅ SQLite DB ready: data/gen_z_survival.db


In [20]:
conn = sqlite3.connect('data/gen_z_survival.db')

QUERIES = {
# rent_to_ppi_ratio: both rent_index and purchasing_power_index are Numbeo indices
# on the same relative scale, so their ratio is a valid affordability signal.
# This replaces the previous formula that divided a Numbeo index by monthly GDP
# (incompatible units that produced a meaningless percentage).
'Affordability — Rent Burden': """
SELECT c.city, c.country, c.region,
       e.cost_of_living_index, e.rent_index, e.purchasing_power_index,
       ROUND(e.rent_index * 1.0 / NULLIF(e.purchasing_power_index, 0), 3) AS rent_to_ppi_ratio,
       CASE WHEN e.rent_index * 1.0 / NULLIF(e.purchasing_power_index, 0) < 0.5 THEN 'Affordable'
            WHEN e.rent_index * 1.0 / NULLIF(e.purchasing_power_index, 0) < 0.8 THEN 'Moderate'
            ELSE 'Expensive' END AS tier
FROM cities c JOIN economic_indicators e ON c.city_id=e.city_id
ORDER BY rent_to_ppi_ratio""",

'AI Opportunity Leaders': """
SELECT c.city, c.country,
       j.ai_job_postings, j.ai_pct_of_total, j.remote_job_ratio, j.data_reliable,
       t.ai_tool_usage_rate, t.chatgpt_usage_pct
FROM cities c
JOIN job_market j ON c.city_id=j.city_id
JOIN tech_adoption t ON c.city_id=t.city_id
ORDER BY j.ai_job_postings DESC LIMIT 15""",

'Frustration vs Cost of Living': """
SELECT c.city, c.region,
       e.cost_of_living_index, e.youth_unemployment_rate,
       l.dev_frustration_pct, l.job_dissatisfaction_pct, l.job_satisfaction_pct,
       ROUND((l.dev_frustration_pct + l.job_dissatisfaction_pct) / 2.0, 2) AS stress_index
FROM cities c
JOIN economic_indicators e ON c.city_id=e.city_id
JOIN lifestyle_metrics l ON c.city_id=l.city_id
ORDER BY stress_index DESC""",

'Regional Tech Snapshot': """
SELECT c.region, COUNT(*) AS cities,
       ROUND(AVG(e.cost_of_living_index),1) AS avg_col,
       ROUND(AVG(e.youth_unemployment_rate),1) AS avg_youth_unemp,
       ROUND(AVG(j.ai_job_postings),0) AS avg_ai_jobs,
       ROUND(AVG(t.ai_tool_usage_rate),1) AS avg_ai_adoption_pct,
       ROUND(AVG(t.remote_hybrid_pct),1) AS avg_remote_hybrid_pct
FROM cities c
JOIN economic_indicators e ON c.city_id=e.city_id
JOIN job_market j ON c.city_id=j.city_id
JOIN tech_adoption t ON c.city_id=t.city_id
GROUP BY c.region ORDER BY avg_ai_adoption_pct DESC""",
}

for name, sql in QUERIES.items():
    df_res = pd.read_sql(sql, conn)
    print(f'\n{"="*60}\n{name}\n{"="*60}')
    print(df_res.head(10).to_string(index=False))
    safe = name.lower().replace(' ','_').replace('—','').replace('/','_').strip('_')
    df_res.to_csv(f'data/processed/{safe}.csv', index=False)

conn.close()
print('\n✅ All query results saved.')



Affordability — Rent Burden
      city     country        region  cost_of_living_index  rent_index  rent_burden_pct       tier
  Montreal      Canada North America                  65.3        29.2              0.6 Affordable
    Dublin     Ireland        Europe                  77.1        60.7              0.6 Affordable
 Melbourne   Australia       Oceania                  77.9        38.0              0.7 Affordable
 Edinburgh          UK        Europe                  61.4        32.5              0.7 Affordable
    Zurich Switzerland        Europe                 120.8        61.8              0.7 Affordable
    Berlin     Germany        Europe                  69.7        38.3              0.8 Affordable
 Stockholm      Sweden        Europe                  70.8        36.5              0.8 Affordable
Copenhagen     Denmark        Europe                  82.3        45.8              0.8 Affordable
    Austin         USA North America                  73.2        66.4          